# DS605 — Lab Assignment 7
## Image Representation, PCA, Eigenfaces and Face Recognition

**Dataset:** AT&T Database of Faces (ORL)  
**Important:** PCA/eigenface calculations below use NumPy linear algebra (`numpy.linalg.eigh` and `numpy.linalg.svd`), not Scikit-learn PCA. Scikit-learn is used only for the traditional classifier comparison.

### Dataset setup
Download the AT&T/ORL face dataset and extract it so the folder contains subject directories such as `s1`, `s2`, …, `s40`, each containing grayscale `.pgm` images. Set `DATASET_DIR` below to the folder containing those subject directories. Common layouts include `att_faces/` and `orl_faces/`.

This notebook is designed to run end-to-end once the dataset is available locally.

In [ ]:
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
DATASET_DIR = Path("att_faces")  # Change this to your extracted ORL dataset folder.
IMAGE_EXTENSIONS = {".pgm", ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

np.random.seed(SEED)
print("Dataset directory:", DATASET_DIR.resolve())

## Part A — Read, represent, and split the images

Each image is converted to grayscale, resized to a consistent shape if needed, and flattened into a vector. The full data matrix has one row per image and one column per pixel. Subject labels are taken from the subject folder names.

In [ ]:
def find_subject_root(root: Path) -> Path:
    if not root.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {root.resolve()}\n"
            "Download/extract the ORL dataset and update DATASET_DIR."
        )
    # Accept a root that directly contains subject folders, or a parent with one nested folder.
    def has_subject_dirs(p):
        return any(d.is_dir() and any(f.suffix.lower() in IMAGE_EXTENSIONS for f in d.iterdir() if f.is_file())
                   for d in p.iterdir())
    if has_subject_dirs(root):
        return root
    candidates = [p for p in root.iterdir() if p.is_dir() and has_subject_dirs(p)]
    if candidates:
        return candidates[0]
    raise FileNotFoundError(
        f"No subject folders with images found under {root.resolve()}. "
        "Expected folders such as s1/1.pgm, s2/1.pgm, etc."
    )

subject_root = find_subject_root(DATASET_DIR)
print("Using subject root:", subject_root.resolve())

image_paths = []
for subject_dir in sorted([p for p in subject_root.iterdir() if p.is_dir()]):
    files = sorted([p for p in subject_dir.iterdir()
                    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])
    for path in files:
        image_paths.append((path, subject_dir.name))

if not image_paths:
    raise RuntimeError("No images found. Check the dataset folder structure.")

# Determine a common image size from the first image.
with Image.open(image_paths[0][0]) as im:
    target_size = im.convert("L").size  # (width, height)

images, labels, paths = [], [], []
for path, label in image_paths:
    with Image.open(path) as im:
        gray = im.convert("L")
        if gray.size != target_size:
            gray = gray.resize(target_size)
        arr = np.asarray(gray, dtype=np.float64)
    images.append(arr)
    labels.append(label)
    paths.append(str(path))

images = np.stack(images)
labels = np.asarray(labels)
n_images, height, width = images.shape
X = images.reshape(n_images, -1)
feature_names = None

print(f"Images: {n_images}")
print(f"Image dimensions: {height} x {width}")
print(f"Features per image: {X.shape[1]}")
print(f"Subjects: {len(np.unique(labels))}")
print("Pixel intensity range:", (X.min(), X.max()))
print("Matrix shape (images x pixels):", X.shape)

In [ ]:
# Inspect a few images and their intensity distributions.
fig, axes = plt.subplots(1, min(5, n_images), figsize=(12, 3))
for ax, idx in zip(np.atleast_1d(axes), range(min(5, n_images))):
    ax.imshow(images[idx], cmap="gray")
    ax.set_title(f"Label: {labels[idx]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3))
plt.hist(X[0], bins=30)
plt.title("Pixel intensity distribution of the first image")
plt.xlabel("Grayscale intensity")
plt.ylabel("Number of pixels")
plt.tight_layout()
plt.show()

In [ ]:
# Reproducible stratified split implemented directly with NumPy.
# Each subject is represented in both train and test sets.
def stratified_split_indices(y, test_fraction=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_idx, test_idx = [], []
    for label in np.unique(y):
        idx = np.flatnonzero(y == label)
        rng.shuffle(idx)
        n_test = max(1, int(round(len(idx) * test_fraction)))
        if n_test >= len(idx):
            n_test = len(idx) - 1
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)
    return np.asarray(train_idx), np.asarray(test_idx)

train_idx, test_idx = stratified_split_indices(labels, test_fraction=0.2, seed=SEED)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = labels[train_idx], labels[test_idx]
paths_train = np.asarray(paths)[train_idx]
paths_test = np.asarray(paths)[test_idx]

print("Train matrix:", X_train.shape, "| Test matrix:", X_test.shape)
print("Subjects in train:", len(np.unique(y_train)))
print("Subjects in test:", len(np.unique(y_test)))
assert set(np.unique(y_train)) == set(np.unique(y_test))

## Part B — PCA, eigenfaces, variance, and compression

PCA is fitted on the training images only. The mean training face is subtracted from every image. For the covariance approach, we use the smaller Gram matrix `X_centered @ X_centered.T`, which avoids constructing a huge pixel-by-pixel covariance matrix. Its eigenvectors are mapped back to pixel space. SVD provides an independent route to the same principal directions (up to sign and numerical differences).

In [ ]:
mean_face = X_train.mean(axis=0)
Xc = X_train - mean_face
n_train, n_features = Xc.shape

# Covariance/eigendecomposition approach (dual/Gram matrix).
t0 = time.perf_counter()
gram = Xc @ Xc.T
eigvals_gram, eigvecs_gram = np.linalg.eigh(gram)
order = np.argsort(eigvals_gram)[::-1]
eigvals_gram = np.maximum(eigvals_gram[order], 0.0)
eigvecs_gram = eigvecs_gram[:, order]

# Convert sample-space eigenvectors to pixel-space principal directions.
eps = np.finfo(float).eps
valid = eigvals_gram > eps * max(1.0, eigvals_gram[0] if eigvals_gram.size else 1.0)
eigvals_cov = eigvals_gram[valid]
U_gram = eigvecs_gram[:, valid]
components_eigh = (Xc.T @ U_gram) / np.sqrt(eigvals_cov)[None, :]
components_eigh = components_eigh.T
time_eigh = time.perf_counter() - t0

# SVD approach.
t0 = time.perf_counter()
U_svd, singular_values, Vt_svd = np.linalg.svd(Xc, full_matrices=False)
components_svd = Vt_svd
eigenvalues_svd = singular_values ** 2
time_svd = time.perf_counter() - t0

# Numerical comparison: compare the subspaces by absolute dot products.
k_compare = min(20, components_eigh.shape[0], components_svd.shape[0])
alignment = np.abs(np.sum(components_eigh[:k_compare] * components_svd[:k_compare], axis=1))
print(f"eigendecomposition time: {time_eigh:.4f} seconds")
print(f"SVD time: {time_svd:.4f} seconds")
print("Mean absolute alignment of first components (sign-invariant):", alignment.mean().round(6))
print("Note: signs of eigenvectors may differ; that does not change the PCA subspace.")

In [ ]:
# Use SVD's ordered singular values for stable explained-variance calculations.
variance = eigenvalues_svd
explained_variance_ratio = variance / variance.sum()
cumulative_variance = np.cumsum(explained_variance_ratio)
n_90 = int(np.searchsorted(cumulative_variance, 0.90) + 1)
n_95 = int(np.searchsorted(cumulative_variance, 0.95) + 1)

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker=".", markersize=3)
plt.axhline(0.90, linestyle="--", label="90% variance")
plt.axvline(n_90, linestyle="--", label=f"{n_90} components")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("Cumulative explained variance")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("Minimum components for >=90% variance:", n_90)
print("Minimum components for >=95% variance:", n_95)

In [ ]:
# Visualize mean face and the first eigenfaces.
mean_img = mean_face.reshape(height, width)
n_eigenfaces = min(10, components_svd.shape[0])
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()
axes[0].imshow(mean_img, cmap="gray")
axes[0].set_title("Mean face")
axes[0].axis("off")
for i in range(1, 10):
    if i - 1 < n_eigenfaces:
        eigface = components_svd[i - 1].reshape(height, width)
        axes[i].imshow(eigface, cmap="gray")
        axes[i].set_title(f"Eigenface {i}")
    axes[i].axis("off")
plt.suptitle("Mean face and principal directions (eigenfaces)")
plt.tight_layout()
plt.show()

In [ ]:
# PCA projection and reconstruction using the minimum component count for 90% variance.
K = n_90
components = components_svd[:K]
Z_train = (X_train - mean_face) @ components.T
Z_test = (X_test - mean_face) @ components.T

def reconstruct_from_pca(Z, mean, components):
    return Z @ components + mean

X_reconstructed = reconstruct_from_pca(Z_test, mean_face, components)
sample_count = min(5, len(X_test))
fig, axes = plt.subplots(2, sample_count, figsize=(3 * sample_count, 5))
for j in range(sample_count):
    axes[0, j].imshow(X_test[j].reshape(height, width), cmap="gray")
    axes[0, j].set_title("Original")
    axes[0, j].axis("off")
    axes[1, j].imshow(np.clip(X_reconstructed[j].reshape(height, width), 0, 255), cmap="gray")
    axes[1, j].set_title(f"Reconstructed, K={K}")
    axes[1, j].axis("off")
plt.tight_layout()
plt.show()

mse = np.mean((X_test - X_reconstructed) ** 2)
rmse = np.sqrt(mse)
print(f"Reconstruction MSE: {mse:.3f}")
print(f"Reconstruction RMSE: {rmse:.3f}")

In [ ]:
# Approximate storage comparison in number of floating-point values.
# Raw storage: all pixels for each image.
# PCA storage: K coefficients per image, plus the shared mean and K component vectors.
n_total = len(X)
raw_values = n_total * n_features
pca_values = n_total * K + n_features + K * n_features
print("Raw representation values:", f"{raw_values:,}")
print("PCA representation + shared model values:", f"{pca_values:,}")
print("Raw/PCA storage ratio (approx.):", round(raw_values / pca_values, 3))
print("Per-image reduction in feature count:", f"{n_features} -> {K}")

## Part C — Recognition using nearest-neighbour similarity

For every test image, find the closest training image by Euclidean distance. We compare the raw pixel representation with the PCA representation. The split is stratified so all subjects appear in both sets.

In [ ]:
def nearest_neighbor_predict(X_train_rep, y_train, X_test_rep, chunk_size=64):
    predictions, nearest_indices = [], []
    for start in range(0, len(X_test_rep), chunk_size):
        batch = X_test_rep[start:start + chunk_size]
        # Squared Euclidean distances, using ||a-b||^2 = ||a||^2 + ||b||^2 - 2a.b
        d2 = (np.sum(batch ** 2, axis=1, keepdims=True)
              + np.sum(X_train_rep ** 2, axis=1)[None, :]
              - 2 * batch @ X_train_rep.T)
        d2 = np.maximum(d2, 0)
        nearest = np.argmin(d2, axis=1)
        nearest_indices.extend(nearest.tolist())
        predictions.extend(y_train[nearest].tolist())
    return np.asarray(predictions), np.asarray(nearest_indices)

y_pred_raw, nn_raw = nearest_neighbor_predict(X_train, y_train, X_test)
y_pred_pca, nn_pca = nearest_neighbor_predict(Z_train, y_train, Z_test)

acc_raw = np.mean(y_pred_raw == y_test)
acc_pca = np.mean(y_pred_pca == y_test)
print(f"Nearest-neighbour accuracy on raw pixels: {acc_raw:.3%}")
print(f"Nearest-neighbour accuracy in PCA space (K={K}): {acc_pca:.3%}")

In [ ]:
# Show some correct and incorrect PCA-space matches.
correct = np.flatnonzero(y_pred_pca == y_test)
incorrect = np.flatnonzero(y_pred_pca != y_test)
chosen = list(correct[:3]) + list(incorrect[:3])
if not chosen:
    print("No test examples to display.")
else:
    fig, axes = plt.subplots(len(chosen), 3, figsize=(9, 3 * len(chosen)))
    if len(chosen) == 1:
        axes = np.array([axes])
    for row, test_i in enumerate(chosen):
        train_i = nn_pca[test_i]
        axes[row, 0].imshow(X_test[test_i].reshape(height, width), cmap="gray")
        axes[row, 0].set_title(f"Test: {y_test[test_i]}")
        axes[row, 1].imshow(X_train[train_i].reshape(height, width), cmap="gray")
        axes[row, 1].set_title(f"Nearest train: {y_train[train_i]}")
        axes[row, 2].axis("off")
        status = "CORRECT" if y_pred_pca[test_i] == y_test[test_i] else "INCORRECT"
        axes[row, 2].text(0.05, 0.7, status, fontsize=13)
        axes[row, 2].text(0.05, 0.4, f"Prediction: {y_pred_pca[test_i]}")
        axes[row, 2].text(0.05, 0.2, f"Actual: {y_test[test_i]}")
        for ax in axes[row, :2]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()
print("Correct PCA matches:", len(correct), "| Incorrect PCA matches:", len(incorrect))

## Part D — Traditional machine-learning classifiers

Two traditional classifiers are trained on PCA features: K-Nearest Neighbours and Support Vector Machine. The PCA representation is created using the training set only; the test set is projected with the same training mean and components.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

knn = KNeighborsClassifier(n_neighbors=1, metric="euclidean")
svm = SVC(kernel="rbf", C=10, gamma="scale")

knn.fit(Z_train, y_train)
svm.fit(Z_train, y_train)
pred_knn = knn.predict(Z_test)
pred_svm = svm.predict(Z_test)

results = [
    ("Nearest neighbour - raw pixels", acc_raw),
    (f"Nearest neighbour - PCA (K={K})", acc_pca),
    (f"KNN on PCA (K={K})", accuracy_score(y_test, pred_knn)),
    (f"SVM on PCA (K={K})", accuracy_score(y_test, pred_svm)),
]
for name, score in results:
    print(f"{name}: {score:.3%}")

print("\nSVM classification report:")
print(classification_report(y_test, pred_svm, zero_division=0))

In [ ]:
# Experiment with different component counts while preserving the same test split.
# The components are selected from the training-only PCA decomposition.
candidate_ks = sorted(set([min(10, len(components_svd)),
                           min(20, len(components_svd)),
                           min(40, len(components_svd)),
                           n_90]))
sweep_rows = []
for k in candidate_ks:
    comp_k = components_svd[:k]
    ztr = (X_train - mean_face) @ comp_k.T
    zte = (X_test - mean_face) @ comp_k.T

    pred_nn, _ = nearest_neighbor_predict(ztr, y_train, zte)
    model_knn = KNeighborsClassifier(n_neighbors=1)
    model_svm = SVC(kernel="rbf", C=10, gamma="scale")
    model_knn.fit(ztr, y_train)
    model_svm.fit(ztr, y_train)

    sweep_rows.append({
        "components": k,
        "explained_variance": float(cumulative_variance[k - 1]),
        "nearest_neighbor_accuracy": float(accuracy_score(y_test, pred_nn)),
        "knn_accuracy": float(accuracy_score(y_test, model_knn.predict(zte))),
        "svm_accuracy": float(accuracy_score(y_test, model_svm.predict(zte))),
    })

import pandas as pd
sweep_df = pd.DataFrame(sweep_rows).drop_duplicates(subset=["components"]).sort_values("components")
display(sweep_df)

## Results and observations (complete after running)

Run all cells with the dataset available, then use the printed metrics and generated figures to complete the observations below. Do not invent results: accuracy and the component count depend on the actual dataset files, split, and library versions.

1. **Dataset:** Record the number of images, subjects, and pixel features.
2. **PCA:** Report the minimum component count needed for at least 90% variance.
3. **Eigenfaces:** Describe the visible patterns in the first few principal directions.
4. **Reconstruction:** Compare original and reconstructed faces and report MSE/RMSE.
5. **Compression:** Compare raw feature storage with the PCA representation plus shared model.
6. **Recognition:** Compare nearest-neighbour accuracy using raw pixels and PCA features.
7. **Classifiers:** Compare KNN and SVM with the similarity baseline using the table above.
8. **Trade-off:** Discuss whether a smaller component count reduces storage while retaining useful recognition accuracy.

### Conclusion template
The ORL face images were converted into grayscale pixel vectors and divided into stratified training and testing sets. PCA was implemented using NumPy linear algebra, with both eigendecomposition and SVD investigated. The cumulative explained-variance curve was used to choose a compact representation retaining at least 90% variance. Face reconstruction and nearest-neighbour recognition were evaluated, and the recognition results were compared with KNN and SVM classifiers. The final conclusion should be updated using the measured values from the run.

## Reproducibility notes
- Random seed: `42`.
- Split: 80% training / 20% testing within each subject, with at least one image per subject in each set.
- Mean face and PCA directions are learned from training images only.
- Recognition accuracy is evaluated on held-out test images.
- The code uses NumPy for PCA/eigenfaces, Pillow for image input, Matplotlib for plots, Pandas for a results table, and Scikit-learn only for the final classifier comparison and reporting.